# Session 1: Introduction to NLP



## Setup
Install (if needed) and import the libraries used throughout this notebook.


In [1]:
# If running for the first time, uncomment the installs below:
# !pip install nltk spacy gensim
# !python -m spacy download en_core_web_sm

import re
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

import spacy

# One-time NLTK data downloads (safe to re-run; skips if already present)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

nlp = spacy.load('en_core_web_sm')

print("NLTK and spaCy ready.")


NLTK and spaCy ready.


---
## Q1. What is NLP? NLU vs. NLG, and Practical Use Cases
What is Natural Language Processing (NLP), and what problem does it solve? Within NLP, distinguish between **Natural Language Understanding (NLU)** and **Natural Language Generation (NLG)** — give one example task for each. List and briefly describe **5 real-world applications** of NLP, noting for each whether it leans more toward NLU, NLG, or uses both.


**Answer:** **Natural Language Processing (NLP)** is the field of AI/computer science concerned with enabling computers to work with human ("natural") language — reading it, making sense of it, and/or producing it — despite language being ambiguous, context-dependent, and unstructured compared to the clean, structured data computers naturally handle. NLP solves the problem of bridging that gap so machines can extract meaning from text/speech and, where needed, generate coherent text/speech in return.

Within NLP, **Natural Language Understanding (NLU)** is the sub-task of taking human language *in* and extracting structured meaning from it — e.g., classifying a review as positive/negative, or extracting a `date` and `destination` entity from "book me a flight to Paris next Friday." **Natural Language Generation (NLG)** is the reverse: taking structured data or an internal representation and producing fluent, human-readable text *out* — e.g., generating a weather report paragraph from raw temperature/forecast data.

**Five real-world applications**, each labeled by whether it leans NLU, NLG, or both (see code output above):
1. **Search engines** — understanding the intent behind a query (NLU) to retrieve/rank relevant results.
2. **Chatbots/virtual assistants** — understanding the user's request (NLU) *and* producing a natural-sounding reply (NLG).
3. **Machine translation** — understanding source-language meaning (NLU) and generating fluent target-language text (NLG).
4. **Sentiment analysis** on reviews/social media — purely understanding/classifying existing text (NLU).
5. **Automatic summarization/report generation** — understanding a long document (NLU) and generating a condensed version (NLG).

---
## Q2. Corpus, Document, Vocabulary, and Tokens
Using the 5-document corpus below, define **Corpus**, **Document**, **Vocabulary**, and **Tokens**, then compute/verify: the number of documents, the total token count across all documents, and the vocabulary size (unique tokens, case-insensitive).


In [ ]:
corpus = [
    "Natural language processing helps computers understand human language.",
    "Tokenization is the first step in most NLP pipelines.",
    "spaCy and NLTK are popular Python libraries for NLP.",
    "Understanding language is hard because language is ambiguous.",
    "Gensim is widely used for topic modeling and word embeddings."
]






In [ ]:
n_docs = len(corpus)

all_tokens = []
for doc in corpus:
    all_tokens.extend(word_tokenize(doc))

total_tokens = len(all_tokens)

In [ ]:
vocabulary = sorted(set(t.lower() for t in all_tokens if t.isalpha()))


In [ ]:
print(f"Number of documents in the corpus: {n_docs}")
print(f"Total tokens across all documents: {total_tokens}")
print(f"Vocabulary size (unique, case-insensitive, alphabetic only): {len(vocabulary)}")
print(f"\nVocabulary:\n{vocabulary}")


## Q3. NLP Libraries: NLTK, spaCy, and Gensim
Compare NLTK, spaCy, and Gensim: design goal, strengths, and a limitation for each. Which would you choose for quickly prototyping a tokenizer in a classroom setting, and which for a production text-classification pipeline?


In [4]:
import nltk as _nltk
import spacy as _spacy
import gensim as _gensim

print("NLTK version:  ", _nltk.__version__)
print("spaCy version: ", _spacy.__version__)
print("Gensim version:", _gensim.__version__)


NLTK version:   3.10.3
spaCy version:  3.8.15
Gensim version: 4.4.0


---
## Q4. Setting Up and Verifying NLTK and spaCy
Confirm your NLTK/spaCy setup by tokenizing the sentence below with both libraries and comparing the outputs.

> "Dr. Smith isn't going to the U.S. next Friday — he's staying home."


In [ ]:
sentence = "Dr. Smith isn't going to the U.S. next Friday — he's staying home."

nltk_tokens = word_tokenize(sentence)
spacy_tokens = [t.text for t in nlp(sentence)]

print("NLTK tokens: ", nltk_tokens)
print("spaCy tokens:", spacy_tokens)
print("\nSame output?", nltk_tokens == spacy_tokens)


---
## Q5. What is Tokenization, and Why Is It Needed?
What is tokenization, and why is it typically the first step in an NLP pipeline? Explain word tokenization vs. sentence tokenization with a short example of each.


In [ ]:
example = "NLP is fascinating. Tokenization comes first!"

word_tokens = word_tokenize(example)


In [ ]:
sentence_tokens = sent_tokenize(example)



In [ ]:
print("Original text:", example)
print("\nWord tokenization ->", word_tokens)
print("Sentence tokenization ->", sentence_tokens)



## Q6. Tokenizing a Paragraph: NLTK vs. spaCy
Tokenize the paragraph below into words with both NLTK and spaCy, and into sentences with both libraries' sentence segmentation. Compare how each handles contractions, punctuation, and the URL/email address.


In [ ]:
paragraph = (
    "NLP is transforming how businesses interact with customers. "
    "Chatbots, don't you think, are getting smarter every year! "
    "Visit https://example.com/nlp for more info, or email us at info@example.com."
)




In [ ]:
# --- Word tokenization ---
nltk_words = word_tokenize(paragraph)
doc = nlp(paragraph)
spacy_words = [t.text for t in doc]

print(f"NLTK word tokens ({len(nltk_words)}):\n{nltk_words}\n")
print(f"spaCy word tokens ({len(spacy_words)}):\n{spacy_words}\n")


In [ ]:
# --- Sentence tokenization ---
nltk_sents = sent_tokenize(paragraph)
spacy_sents = [s.text for s in doc.sents]

print("NLTK sentences:")
for s in nltk_sents:
    print(" -", s)

print("\nspaCy sentences:")
for s in spacy_sents:
    print(" -", s)



## Q7. Building a Vocabulary and Token Frequencies
Using the corpus from Q2, tokenize every document (lowercased, punctuation removed), build the vocabulary, and compute token frequency across the whole corpus. Print the top 10 most frequent tokens, then remove stopwords and recompute. How does the ranking change, and why remove stopwords first?


In [ ]:
def simple_tokenize(text):
    """Lowercase and keep only alphabetic word tokens (drops punctuation)."""
    return re.findall(r"\b[a-z]+\b", text.lower())

all_tokens_q7 = []
for doc in corpus:
    all_tokens_q7.extend(simple_tokenize(doc))




In [ ]:
freq_all = Counter(all_tokens_q7)
print("Top 10 tokens WITH stopwords:")
for word, count in freq_all.most_common(10):
    print(f"  {word:15s} {count}")



In [ ]:
stop_words = set(stopwords.words('english'))
filtered_tokens = [t for t in all_tokens_q7 if t not in stop_words]
freq_filtered = Counter(filtered_tokens)



In [ ]:
print("\nTop 10 tokens WITHOUT stopwords:")
for word, count in freq_filtered.most_common(10):
    print(f"  {word:15s} {count}")


## Q8. Tokenization Edge Cases
Construct text containing a contraction, a hashtag, a URL/email, and an emoji/emoticon. Tokenize it with both NLTK and spaCy, and note how each handles each edge case. Which library handles messy, real-world text more gracefully?


In [ ]:
edge_text = "I can't believe #NLProc is this fun :) Check https://nlp.example.com or email me@example.com!"

nltk_edge_tokens = word_tokenize(edge_text)
spacy_edge_tokens = [t.text for t in nlp(edge_text)]

print(f"NLTK tokens ({len(nltk_edge_tokens)}):\n{nltk_edge_tokens}\n")
print(f"spaCy tokens ({len(spacy_edge_tokens)}):\n{spacy_edge_tokens}")
